## 🧩 Conceptual Background: Scaling vs Semantic Consistency in VRPTW

**Author:** Miguel Vásquez  
**Date:** October 2025  
**Role:** Data Engineer | Machine Learning Engineer  

---

## 1. Introduction

In traditional machine learning workflows, feature scaling (e.g., MinMax or StandardScaler) is a common preprocessing step to stabilize gradient-based convergence and ensure equal feature contribution.  

However, in **combinatorial optimization problems** such as the **Vehicle Routing Problem with Time Windows (VRPTW)**, scaling must be treated with caution due to its **semantic implications** — i.e., the risk of distorting the real-world meaning of time, distance, and capacity constraints.

---

## 2. Scaling in ML vs Optimization

| Context | Purpose of Scaling | Impact if Misused |
|----------|--------------------|-------------------|
| **Machine Learning** | Stabilizes model training and prevents feature dominance | None — values are abstract representations |
| **VRPTW / Optimization** | Should preserve physical relationships (distance, time, capacity) | Alters cost structures and route feasibility |

In optimization, every numeric value encodes a *real-world constraint*. Scaling distances, times, or capacities arbitrarily modifies the problem itself — not just the representation.

---

## 3. Why Arbitrary Normalization Can Be Harmful

When coordinates or time windows are normalized to a 0–1 range, the solver loses physical context.  
Example:

| Customer | X | Y | Ready_Time | Due_Date |
|-----------|---|---|-------------|-----------|
| 1 | 10 | 20 | 100 | 200 |
| 2 | 30 | 40 | 300 | 400 |

- **Real distance:** √((30−10)² + (40−20)²) = 28.3  
- **Normalized distance (0–1):** √(1² + 1²) = 1.41  

This scaling drastically alters relative distances and time costs, leading to **incorrect route feasibility** and **biased cost optimization**.

---

## 4. Objectives for this Notebook

1. Maintain semantic consistency across spatial (`XCOORD`, `YCOORD`), temporal (`READY_TIME`, `DUE_DATE`), and capacity (`DEMAND`) variables.  
2. Ensure consistent units across all Solomon instances.  
3. Prepare reliable **distance and time matrices** that reflect the real geometry and timing of the problem.  
4. Provide reusable data structures for both **exact solvers (e.g., OR-Tools)** and **heuristic/metaheuristic** methods.

---

## 5. Recommended Practices

Instead of arbitrary normalization:
- ✅ **Ensure consistent units** (e.g., all times in minutes, all distances in coordinate units).  
- ✅ **Standardize instance scales** if datasets come from different domains (e.g., divide coordinates by 100 *only if* all instances use that convention).  
- ❌ **Avoid normalization to 0–1** or z-scores for solver input data.

Scaling may still be applied for:
- Visual analysis (e.g., plotting customer distribution),
- Exploratory correlation studies,
- Diagnostic comparisons between instances.

But **never directly** in solver execution.

---

## 6. Implementation Note

In this project, normalization is retained **only for diagnostic and comparative purposes**, not as an input to the optimization engine.  
The solver will operate on **original-scale data**, ensuring that all spatial, temporal, and capacity constraints remain physically meaningful.

> In optimization, **the numbers represent the world**.  
> In machine learning, **the numbers represent relationships**.

---

## 7. Generate Distance & Time Matrices

In this section, we compute the **core matrices** required by the VRPTW solver:

- **Distance Matrix** — Euclidean distances between all customer nodes.
- **Travel Time Matrix** — Derived from distances assuming a uniform travel speed.
- **Service Time Integration** — Unique per instance, ensuring cross-instance compatibility. 

These matrices will serve as the foundation for constraint-based solvers (e.g., OR-Tools, heuristics, or metaheuristics).

### 7.1 Load Preprocessed Data
We start by loading the normalized dataset prepared in the previous notebook (`02_data_preparation.ipynb`).

In [1]:
import sys
import numpy as np
import pandas as pd
from math import sqrt
from pathlib import Path
from IPython.display import Markdown, display

# Add the parent directory to sys.path so 'src' can be imported
sys.path.append(str(Path.cwd().parent))

from src.data_utils import ensure_project_root, load_processed_data

# Ensure we're in the project root
ensure_project_root()

# Load preprocessed dataset
processed_path = "data/processed/vrptw_ready_for_optimization.csv"
df = load_processed_data(processed_path)
display(df.head())

Changed working directory to project root: `c:\Users\Miguel\portfolio\Route-Optimization-VRPTW`

Loading processed dataset from: data/processed/vrptw_ready_for_optimization.csv
✅ Loaded dataset with 5656 rows and 10 columns.


,CUST_NO,XCOORD,YCOORD,DEMAND,READY_TIME,DUE_DATE,SERVICE_TIME,INSTANCE,TYPE,TYPE_ENCODED
0,1,0.421053,0.573171,0.0,0.000000,0.364602,0.000000,C101,C,0
1,2,0.473684,0.792683,0.2,0.269027,0.285251,0.026549,C101,C,0
2,3,0.473684,0.817073,0.6,0.243363,0.256637,0.026549,C101,C,0
3,4,0.442105,0.768293,0.2,0.019174,0.043068,0.026549,C101,C,0
4,5,0.442105,0.792683,0.2,0.214454,0.230678,0.026549,C101,C,0


### 7.2 Ensure Unique Node Identifiers
Since customer numbers repeat across Solomon instances (e.g., multiple `CUST_NO = 1`), we create a unique identifier per node combining instance and customer number.

In [ ]:
# Ensure unique node identifiers
df["NODE_ID"] = df["INSTANCE"] + "_" + df["CUSTOMER"].astype(str)

# Validate uniqueness (critical for matrix integrity)
if not df["NODE_ID"].is_unique:
    raise ValueError("❌ Duplicate NODE_ID values detected! Each node must be unique across all instances.")
else:
    print("✅ All NODE_ID values are unique.")

# Extract coordinates
coords = df[["XCOORD", "YCOORD"]].to_numpy()

df.head()

✅ All NODE_ID values are unique.


,CUST_NO,XCOORD,YCOORD,DEMAND,READY_TIME,DUE_DATE,SERVICE_TIME,INSTANCE,TYPE,TYPE_ENCODED,NODE_ID
0,1,0.421053,0.573171,0.0,0.000000,0.364602,0.000000,C101,C,0,C101_1
1,2,0.473684,0.792683,0.2,0.269027,0.285251,0.026549,C101,C,0,C101_2
2,3,0.473684,0.817073,0.6,0.243363,0.256637,0.026549,C101,C,0,C101_3
3,4,0.442105,0.768293,0.2,0.019174,0.043068,0.026549,C101,C,0,C101_4
4,5,0.442105,0.792683,0.2,0.214454,0.230678,0.026549,C101,C,0,C101_5


### 7.3 Compute Euclidean Distance Matrix
We compute pairwise distances between every pair of nodes using standard Euclidean distance.

In [3]:
distance_matrix = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2))

# Build DataFrame for easier inspection
distance_df = pd.DataFrame(distance_matrix, index=df["NODE_ID"], columns=df["NODE_ID"])

# Remove axis names for cleaner visual display
distance_df.index.name = None
distance_df.columns.name = None

# Diagnostics
display(Markdown("### Euclidean Distance Matrix Generated"))
display(Markdown(f"- Shape: **{distance_df.shape}** (Nodes × Nodes)"))
display(Markdown(f"- Example subset (first 10×10):"))
display(distance_df.iloc[:10, :10])

### Euclidean Distance Matrix Generated

- Shape: **(5656, 5656)** (Nodes × Nodes)

- Example subset (first 10×10):

,C101_1,C101_2,C101_3,C101_4,C101_5,C101_6,C101_7,C101_8,C101_9,C101_10
C101_1,0.000000,0.225734,0.249516,0.196254,0.220519,0.184134,0.231707,0.195122,0.220519,0.244809
C101_2,0.225734,0.000000,0.024390,0.039901,0.031579,0.048329,0.054026,0.058008,0.073684,0.077616
C101_3,0.249516,0.024390,0.000000,0.058110,0.039901,0.068668,0.054026,0.071761,0.077616,0.073684
C101_4,0.196254,0.039901,0.058110,0.000000,0.024390,0.012195,0.042210,0.021053,0.048659,0.064439
C101_5,0.220519,0.031579,0.039901,0.024390,0.000000,0.036585,0.024330,0.032220,0.042105,0.048659
C101_6,0.184134,0.048329,0.068668,0.012195,0.036585,0.000000,0.053130,0.024330,0.055779,0.074100
C101_7,0.231707,0.054026,0.054026,0.042210,0.024330,0.053130,0.000000,0.036585,0.024330,0.024330
C101_8,0.195122,0.058008,0.071761,0.021053,0.032220,0.024330,0.036585,0.000000,0.032220,0.053130
C101_9,0.220519,0.073684,0.077616,0.048659,0.042105,0.055779,0.024330,0.032220,0.000000,0.024390
C101_10,0.244809,0.077616,0.073684,0.064439,0.048659,0.074100,0.024330,0.053130,0.024390,0.000000


### 7.4 Compute Travel Time Matrix
Assuming a **constant vehicle speed** (e.g., 1 unit distance = 1 unit time), the travel time matrix equals the distance matrix.
If you have speed data later, this formula can easily be adjusted.

In [4]:
VEHICLE_SPEED = 1.0 
time_matrix = distance_df / VEHICLE_SPEED

display(Markdown(f"Time matrix computed (speed = {VEHICLE_SPEED} units/time)."))
display(time_matrix.head(5))

Time matrix computed (speed = 1.0 units/time).

,C101_1,C101_2,C101_3,C101_4,C101_5,C101_6,C101_7,C101_8,C101_9,C101_10,...,RC208_92,RC208_93,RC208_94,RC208_95,RC208_96,RC208_97,RC208_98,RC208_99,RC208_100,RC208_101
C101_1,0.000000,0.225734,0.249516,0.196254,0.220519,0.184134,0.231707,0.195122,0.220519,0.244809,...,0.135990,0.161286,0.222394,0.180602,0.231300,0.165258,0.543959,0.149373,0.234904,0.227937
C101_2,0.225734,0.000000,0.024390,0.039901,0.031579,0.048329,0.054026,0.058008,0.073684,0.077616,...,0.319857,0.316294,0.257756,0.274671,0.395383,0.200573,0.747036,0.279415,0.449396,0.147872
C101_3,0.249516,0.024390,0.000000,0.058110,0.039901,0.068668,0.054026,0.071761,0.077616,0.073684,...,0.344050,0.339866,0.276679,0.296541,0.418765,0.221705,0.767074,0.296961,0.471363,0.151842
C101_4,0.196254,0.039901,0.058110,0.000000,0.024390,0.012195,0.042210,0.021053,0.048659,0.064439,...,0.301816,0.303448,0.262963,0.270400,0.383134,0.200354,0.708980,0.239823,0.413868,0.116430
C101_5,0.220519,0.031579,0.039901,0.024390,0.000000,0.036585,0.024330,0.032220,0.042105,0.048659,...,0.325522,0.326125,0.279415,0.290550,0.405756,0.218804,0.729248,0.257756,0.436260,0.116430


### 7.5 Save Generated Matrices
Save both matrices for later solver use.
These artifacts will be consumed by the optimization notebook (`04_solver_vrptw.ipynb`).

In [5]:
from src.data_utils import save_processed_data

# Save matrices
output_path_distance = save_processed_data(distance_df, "vrptw_distance_matrix.csv")
output_path_time = save_processed_data(time_matrix, "vrptw_time_matrix.csv")

✅ Data saved successfully at: data/processed\vrptw_distance_matrix.csv
Rows: 5656 | Columns: 5656
✅ Data saved successfully at: data/processed\vrptw_time_matrix.csv
Rows: 5656 | Columns: 5656


## 7. Summary & Next Steps

In this notebook, we successfully generated the **core spatial and temporal structures** required for the VRPTW solver:
- Constructed the **Euclidean distance matrix** for all customer nodes across instances.
- Derived a **time matrix** assuming a baseline travel speed (1 unit of distance per unit of time).
- Ensured all identifiers were unique and consistently formatted.
- Exported the matrices to the `data/processed/` directory for downstream use.
These matrices now serve as the **foundational inputs** for optimization modeling and scenario simulation.

---

### Next Steps — Toward Realistic Scenario Modeling
In the next notebook, `04_scenario_generation_and_robustness_analysis.ipynb`, we will:
- Introduce **stochastic variations** to simulate realistic logistics conditions (e.g., variable speeds, congestion, accidents).
- Analyze how time and distance perturbations affect route feasibility and total cost.
- Prepare multiple scenario matrices that will feed into the solver’s **robust optimization** stage.